# 04 - Chunking and Splitting

In [1]:
!pip install -qU --break-system-packages langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 3.0 MB/s eta 0:00:00


In [2]:
import json
import re
import hashlib
from pathlib import Path
from collections import Counter
from langchain_text_splitters import RecursiveCharacterTextSplitter

## 1. Konfigurasi

In [3]:
PIPELINE_VERSION = "chunking_v3_rag"
CHUNK_SIZE = 2000
CHUNK_OVERLAP = 200
 
STRUCTURED_DIR = Path("/kaggle/input/notebooks/muhammadnabilul/03-structured-cleaned-extract/structured")
QUALITY_REPORT_PATH = Path("/kaggle/input/notebooks/muhammadnabilul/01-extract-pdf-pymupdfllm-ocr/extracted/quality_report.json")
OUTPUT_CHUNKS_DIR = Path("/kaggle/working/chunks")
OUTPUT_CHUNKS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_REPORT_DIR = Path("/kaggle/working/laporan")
OUTPUT_REPORT_DIR.mkdir(parents=True, exist_ok=True)
 
QA_CANDIDATES = {
    "mixed_rich":  ["statistical", "comparative", "interpretative"],
    "table_rich":  ["statistical", "comparative"],
    "figure_rich": ["interpretative", "comparative", "trend"],
    "text_only":   ["definition", "factual", "summary"],
    "toc_table":   [],
    "toc_figure":  [],
}
 
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", " ", ""],
)

## 2. Filter Noise

In [4]:
TOC_TABLE_PATTERN = re.compile(r"daftar\s+tabel|list\s+of\s+tables", re.IGNORECASE)
TOC_FIGURE_PATTERN = re.compile(r"daftar\s+(gambar|grafik)|list\s+of\s+figures", re.IGNORECASE)
 
FRONT_MATTER_PATTERN = re.compile(
    r"(tim\s+penyusun|compilers|kata\s+pengantar|preface|daftar\s+isi|contents|"
    r"singkatan|abbreviation|acronyms|penjelasan\s+umum|explanation\s+notes|"
    r"katalog|catalogue|issn|nomor\s+publikasi|publication\s+number|ukuran\s+buku|book\s+size|"
    r"^pengarah|director|penanggung\s+jawab|persons\s+in\s+charge|^penyunting|editors|"
    r"penulis\s+naskah|writers|ucapan\s+terima\s+kasih|acknowledg|prolog|"
    r"ringkasan\s+eksekutif|executive\s+summary)",
    re.IGNORECASE,
)
 
REFERENCE_PATTERN = re.compile(
    r"^\s*[*_#\s]*(daftar\s+pustaka|daftar\s+referensi|referensi|references|bibliografi|"
    r"bibliography|pustaka\s+acuan|lampiran|appendix|glosarium|glossary)[*_\s]*$",
    re.IGNORECASE,
)
 
NOISE_TITLE_PATTERNS = [
    r"^\s*[a-zA-Z]\.\s*$",
    r"^\s*\d+\.\s*$",
    r"^\s*\*{0,4}\s*$",
]
 
DOTTED_LEADER = re.compile(r"\.{4,}")
OCR_BROKEN_RATIO = 0.45
OCR_MIN_TOKENS = 6
 
exclusion_log = []

In [5]:
def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)
 
 
def save_json(data, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
 
 
def norm_title(title):
    return re.sub(r"[*_`#]", "", title or "").strip()
 
 
def section_kind(section):
    title = norm_title(section.get("title", ""))
    if REFERENCE_PATTERN.match(title):
        return "reference"
    if TOC_TABLE_PATTERN.search(title):
        return "toc_table"
    if TOC_FIGURE_PATTERN.search(title):
        return "toc_figure"
    if FRONT_MATTER_PATTERN.search(title):
        return "front_matter"
    return "content"
 
 
def ocr_broken_ratio(text):
    tokens = text.split()
    if len(tokens) < OCR_MIN_TOKENS:
        return 0.0
    short = sum(1 for t in tokens if re.search(r"[A-Za-z]", t) and len(re.sub(r"[^A-Za-z]", "", t)) <= 2)
    return short / len(tokens)
 
 
def is_ocr_broken(text):
    lines = [ln for ln in text.split("\n") if ln.strip()]
    if not lines:
        return False
    bad = sum(1 for ln in lines if ocr_broken_ratio(ln) >= OCR_BROKEN_RATIO)
    return bad / len(lines) >= 0.5

## 3. Fungsi-Fungsi

In [6]:
def clean_toc_text(text):
    text = DOTTED_LEADER.sub(" ", text)
    text = re.sub(r"\|+", " ", text)
    text = re.sub(r"-{3,}", " ", text)
    text = re.sub(r"<br\s*/?>", " ", text)
    text = re.sub(r"\*{0,4}=+>?\s*picture[^<]*<==\*{0,4}", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"-{2,}\s*start of picture text\s*-{2,}", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"[ \t]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

In [7]:
def clean_block_text(text):
    text = re.sub(r"^#{1,6}\s*", "", text, flags=re.MULTILINE)
    text = re.sub(r"\*{1,3}([^*\n]+)\*{1,3}", r"\1", text)
    text = re.sub(r"_{1,2}([^_\n]+)_{1,2}", r"\1", text)
    text = re.sub(r"`([^`]+)`", r"\1", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

In [8]:
def merge_tiny_chunks(sub_texts):
    merged = []
    buffer = ""
    for text in sub_texts:
        lines = text.strip().splitlines()
        combined = (buffer + "\n\n" + text).strip() if buffer else text
        if len(lines) <= 2 and len(text.split()) <= 10:
            buffer = combined
        else:
            merged.append(combined)
            buffer = ""
    if buffer:
        if merged:
            merged[-1] += "\n\n" + buffer
        else:
            merged.append(buffer)
    return merged
 
 
def build_section_text_with_map(blocks, sep="\n\n"):
    parts = []
    page_map = []
    cursor = 0
    first = True
    for b in blocks:
        ct = clean_block_text(b.get("text", ""))
        if not ct:
            continue
        if not first:
            cursor += len(sep)
        start = cursor
        end = cursor + len(ct)
        page_map.append((start, end, b.get("page_number"), b.get("chunk_id")))
        parts.append(ct)
        cursor = end
        first = False
    return sep.join(parts), page_map
 
 
def provenance_for_span(cstart, cend, page_map):
    pages = set()
    chunk_ids = set()
    for s, e, pg, cid in page_map:
        if cstart < e and cend > s:
            if pg is not None:
                pages.add(pg)
            if cid:
                chunk_ids.add(cid)
    return sorted(pages), sorted(chunk_ids)
 
 
def is_garbage_chunk(chunk):
    text = chunk["text"].strip()
    title = chunk["section_title"].strip()
    if not text:
        return True
    if text.lower() == title.lower():
        return True
    if re.match(r"^.*\n\d+$", text):
        return True
    if (
        chunk["word_count"] < 10
        and chunk["table_count"] == 0
        and chunk["figure_count"] == 0
        and "." not in text
    ):
        return True
    return False

In [9]:
def log_exclusion(base_name, section, reason, detail=""):
    exclusion_log.append({
        "source_file": base_name,
        "section_id": section.get("section_id"),
        "section_title": norm_title(section.get("title", "")),
        "page_start": section.get("page_start"),
        "reason": reason,
        "detail": detail,
    })

In [10]:
def build_toc_chunks(doc, section, kind):
    metadata = doc.get("document_metadata", {})
    base_name = metadata.get("nama_file", "doc").replace(".pdf", "")
    blocks = section.get("blocks", [])
    raw = "\n\n".join(b.get("text", "") for b in blocks if b.get("text", "").strip())
    cleaned = clean_toc_text(raw)
    if not cleaned.strip():
        return []
 
    raw_sid = str(section.get("section_id", "toc"))
    section_id = re.sub(r"[^a-zA-Z0-9_]", "_", raw_sid)
    section_title = norm_title(section.get("title", "No Title"))
    source_pages = sorted(set(b.get("page_number") for b in blocks if b.get("page_number") is not None))
 
    sub_texts = text_splitter.split_text(cleaned)
    chunks = []
    for i, text in enumerate(sub_texts):
        chunk = {
            "chunk_id": f"{base_name}__{section_id}__toc{i+1:03d}",
            "context_hash": hashlib.sha256(text.encode()).hexdigest(),
            "pipeline_version": PIPELINE_VERSION,
            "document_title": metadata.get("judul"),
            "document_year": metadata.get("tahun"),
            "source_file": metadata.get("nama_file"),
            "bps_url": metadata.get("bps_url"),
            "section_id": section_id,
            "section_title": section_title,
            "section_level": section.get("level", 0),
            "section_source": section.get("source", "unknown"),
            "page_start": section.get("page_start"),
            "page_end": section.get("page_end"),
            "chunk_page_start": min(source_pages) if source_pages else section.get("page_start"),
            "chunk_page_end": max(source_pages) if source_pages else section.get("page_end"),
            "source_pages": source_pages,
            "source_chunk_ids": sorted(set(b.get("chunk_id") for b in blocks if b.get("chunk_id"))),
            "chunk_index": i + 1,
            "total_chunks_in_section": len(sub_texts),
            "is_first_chunk": (i == 0),
            "is_last_chunk": (i == len(sub_texts) - 1),
            "table_count": 0,
            "figure_count": 0,
            "chunk_type": kind,
            "qa_candidate": QA_CANDIDATES[kind],
            "estimated_qa_count": 0,
            "provenance_fallback": False,
            "text": text,
            "char_count": len(text),
            "word_count": len(text.split()),
            "token_estimate": int(len(text.split()) * 1.3),
        }
        chunks.append(chunk)
    return chunks

## 4. Main Program

In [11]:
def build_master_chunks(doc):
    metadata = doc.get("document_metadata", {})
    base_name = metadata.get("nama_file", "doc").replace(".pdf", "")
    all_chunks = []
 
    for idx, section in enumerate(doc.get("sections", [])):
        kind = section_kind(section)
 
        if kind == "reference":
            log_exclusion(base_name, section, "reference")
            continue
        if kind == "front_matter":
            log_exclusion(base_name, section, "front_matter")
            continue
 
        blocks = section.get("blocks", [])
        if not blocks:
            log_exclusion(base_name, section, "empty_section")
            continue
 
        raw_title = norm_title(section.get("title", "")).lower()
        page_start = section.get("page_start", 999)
        if page_start <= 15:
            if any(re.search(p, raw_title, re.IGNORECASE) for p in NOISE_TITLE_PATTERNS):
                log_exclusion(base_name, section, "noise_title")
                continue
 
        if kind in ("toc_table", "toc_figure"):
            toc_chunks = build_toc_chunks(doc, section, kind)
            all_chunks.extend(toc_chunks)
            continue
 
        seen_ids = set()
        unique_blocks = []
        for b in blocks:
            cid = b.get("chunk_id", "")
            if cid not in seen_ids:
                seen_ids.add(cid)
                unique_blocks.append(b)
 
        section_text, page_map = build_section_text_with_map(unique_blocks)
 
        if not section_text.strip():
            log_exclusion(base_name, section, "empty_after_clean")
            continue
 
        if is_ocr_broken(section_text):
            log_exclusion(base_name, section, "ocr_broken", detail=section_text[:120])
            continue
 
        raw_sid = str(section.get("section_id", f"sec_{idx:03d}"))
        section_id = re.sub(r"[^a-zA-Z0-9_]", "_", raw_sid)
        section_title = norm_title(section.get("title", "No Title"))
 
        all_section_pages = sorted(set(p for _, _, p, _ in page_map if p is not None))
        all_section_cids = sorted(set(c for _, _, _, c in page_map if c))
 
        sub_texts = text_splitter.split_text(section_text)
        final_texts = merge_tiny_chunks(sub_texts)
        total_sub = len(final_texts)
 
        search_cursor = 0
        for i, text in enumerate(final_texts):
            probe = text[:80].strip()
            pos = section_text.find(probe, search_cursor) if probe else -1
            if pos < 0:
                pos = section_text.find(probe) if probe else -1
            if pos < 0:
                cstart, cend = search_cursor, search_cursor + len(text)
                fallback = True
            else:
                cstart, cend = pos, pos + len(text)
                search_cursor = pos + 1
                fallback = False
 
            source_pages, source_chunk_ids = provenance_for_span(cstart, cend, page_map)
 
            if not source_pages:
                source_pages = all_section_pages
                fallback = True
            if not source_chunk_ids:
                source_chunk_ids = all_section_cids
 
            tl = text.lower()
            sec_table_count = len(section.get("tables", []))
            sec_figure_count = len(section.get("figures", []))
 
            has_table = (bool(re.search(r"\btabel\s+[a-z0-9\.]+", tl)) or ("|" in text and "\n|" in text))
            has_figure = bool(re.search(r"\b(gambar|grafik|diagram)\s+[a-z0-9\.]+", tl))
 
            if has_table and has_figure and (sec_table_count > 0 or sec_figure_count > 0):
                chunk_type = "mixed_rich"
            elif has_table and sec_table_count > 0:
                chunk_type = "table_rich"
            elif has_figure and sec_figure_count > 0:
                chunk_type = "figure_rich"
            else:
                chunk_type = "text_only"
 
            chunk = {
                "chunk_id": f"{base_name}__{section_id}__c{i+1:03d}",
                "context_hash": hashlib.sha256(text.encode()).hexdigest(),
                "pipeline_version": PIPELINE_VERSION,
                "document_title": metadata.get("judul"),
                "document_year": metadata.get("tahun"),
                "source_file": metadata.get("nama_file"),
                "bps_url": metadata.get("bps_url"),
                "section_id": section_id,
                "section_title": section_title,
                "section_level": section.get("level", 0),
                "section_source": section.get("source", "unknown"),
                "page_start": section.get("page_start"),
                "page_end": section.get("page_end"),
                "chunk_page_start": min(source_pages) if source_pages else section.get("page_start"),
                "chunk_page_end": max(source_pages) if source_pages else section.get("page_end"),
                "source_pages": source_pages,
                "source_chunk_ids": source_chunk_ids,
                "chunk_index": i + 1,
                "total_chunks_in_section": total_sub,
                "is_first_chunk": (i == 0),
                "is_last_chunk": (i == total_sub - 1),
                "table_count": sec_table_count,
                "figure_count": sec_figure_count,
                "chunk_type": chunk_type,
                "qa_candidate": QA_CANDIDATES[chunk_type],
                "estimated_qa_count": len(QA_CANDIDATES[chunk_type]),
                "provenance_fallback": fallback,
                "text": text,
                "char_count": len(text),
                "word_count": len(text.split()),
                "token_estimate": int(len(text.split()) * 1.3),
            }
 
            if is_garbage_chunk(chunk):
                exclusion_log.append({
                    "source_file": base_name,
                    "section_id": section_id,
                    "section_title": section_title,
                    "page_start": section.get("page_start"),
                    "reason": "garbage_chunk",
                    "chunk_type": chunk["chunk_type"],
                    "word_count": chunk["word_count"],
                    "table_count": chunk["table_count"],
                    "figure_count": chunk["figure_count"],
                    "detail": text[:150],
                })
                continue
            all_chunks.append(chunk)
 
    return all_chunks

In [12]:
structured_files = sorted(STRUCTURED_DIR.glob("*.json"))
print(f"Menjalankan {PIPELINE_VERSION} untuk {len(structured_files)} dokumen BPS...\n")
 
total_chunks = 0
chunk_type_counter = Counter()
fallback_count = 0
per_doc_summary = []
 
for file_path in structured_files:
    doc = load_json(file_path)
    chunks = build_master_chunks(doc)
    total_chunks += len(chunks)
 
    for c in chunks:
        chunk_type_counter[c["chunk_type"]] += 1
        if c.get("provenance_fallback"):
            fallback_count += 1
 
    out_path = OUTPUT_CHUNKS_DIR / f"{file_path.stem}_chunks.json"
    save_json(chunks, out_path)
 
    n_sections = len(doc.get("sections", []))
    per_doc_summary.append({
        "source_file": doc.get("document_metadata", {}).get("nama_file"),
        "n_sections": n_sections,
        "n_chunks": len(chunks),
        "n_toc_chunks": sum(1 for c in chunks if c["chunk_type"].startswith("toc")),
    })
    print(f"  {file_path.name}: {n_sections} sections -> {len(chunks)} chunks "
          f"(toc={sum(1 for c in chunks if c['chunk_type'].startswith('toc'))})")
 
exclusion_counter = Counter(e["reason"] for e in exclusion_log)
 
with open(OUTPUT_REPORT_DIR / "excluded_blocks.jsonl", "w", encoding="utf-8") as f:
    for e in exclusion_log:
        f.write(json.dumps(e, ensure_ascii=False) + "\n")

Menjalankan chunking_v3_rag untuk 47 dokumen BPS...

  analisis-petty-corruption-hasil-spak-2020---2024.json: 115 sections -> 374 chunks (toc=9)
  indeks-pembangunan-manusia-2022.json: 230 sections -> 293 chunks (toc=4)
  indeks-pembangunan-manusia-2023.json: 125 sections -> 268 chunks (toc=20)
  indeks-pembangunan-manusia-2024.json: 186 sections -> 336 chunks (toc=12)
  indeks-perilaku-anti-korupsi-2023.json: 94 sections -> 189 chunks (toc=10)
  indeks-perilaku-anti-korupsi-2024.json: 117 sections -> 229 chunks (toc=9)
  indikator-kesejahteraan-rakyat-2023.json: 226 sections -> 510 chunks (toc=11)
  indikator-kesejahteraan-rakyat-2024.json: 175 sections -> 516 chunks (toc=7)
  indikator-kesejahteraan-rakyat-2025.json: 139 sections -> 270 chunks (toc=6)
  indikator-sdgs-kesejahteraan-rakyat-2024.json: 162 sections -> 374 chunks (toc=2)
  indikator-sdgs-kesejahteraan-rakyat-2025.json: 209 sections -> 328 chunks (toc=2)
  infografis-statistik-kesejahteraan-rakyat-2025.json: 68 sections -

## 5. Ringkasan

In [13]:
print("\n" + "=" * 64)
print("RINGKASAN CHUNKING")
print("=" * 64)
print(f"\nTotal chunks RAG : {total_chunks:,}")
print(f"\nDistribusi chunk_type:")
for k, v in chunk_type_counter.most_common():
    print(f"  {k:<12}: {v:>6} ({v/total_chunks*100:.1f}%)")
 
print(f"\nProvenance fallback (source_pages tidak presisi): {fallback_count:,} "
      f"({fallback_count/max(total_chunks,1)*100:.1f}%)")
 
print(f"\nSection disisihkan (total {len(exclusion_log)}):")
for k, v in exclusion_counter.most_common():
    print(f"  {k:<18}: {v:>4}")
 
garbage = [e for e in exclusion_log if e["reason"] == "garbage_chunk"]
if garbage:
    print("\n" + "=" * 64)
    print(f"EVALUASI GARBAGE_CHUNK ({len(garbage)} chunk)")
    print("=" * 64)
    g_by_type = Counter(e.get("chunk_type", "?") for e in garbage)
    g_with_table = sum(1 for e in garbage if e.get("table_count", 0) > 0 or e.get("figure_count", 0) > 0)
    print(f"\n  Per chunk_type: {dict(g_by_type)}")
    print(f"  Punya tabel/gambar di section (suspek valid): {g_with_table}")
    wc = [e.get("word_count", 0) for e in garbage]
    print(f"  Word count: min={min(wc)} median={sorted(wc)[len(wc)//2]} max={max(wc)}")
    print(f"\n  30 contoh teks garbage:")
    for e in garbage[:30]:
        print(f"    [{e.get('word_count'):>2}w tbl={e.get('table_count',0)}] {e['detail'][:90]!r}")
 
quality = None
if QUALITY_REPORT_PATH.exists():
    quality = load_json(QUALITY_REPORT_PATH)
 
final_report = {
    "pipeline_version": PIPELINE_VERSION,
    "n_documents": len(structured_files),
    "total_chunks": total_chunks,
    "chunk_type_distribution": dict(chunk_type_counter),
    "provenance_fallback_count": fallback_count,
    "provenance_fallback_pct": round(fallback_count / max(total_chunks, 1) * 100, 2),
    "exclusion_summary": dict(exclusion_counter),
    "per_document": per_doc_summary,
    "extraction_quality_source": "quality_report_tipe_2",
}
 
if quality:
    total_pages = sum(d.get("total_halaman", 0) for d in quality)
    usable_pages = sum(d.get("halaman_usable", 0) for d in quality)
    pages_tables = sum(d.get("layout_stats", {}).get("pages_with_tables", 0) for d in quality)
    pages_pictures = sum(d.get("layout_stats", {}).get("pages_with_pictures", 0) for d in quality)
    final_report["extraction_summary"] = {
        "total_halaman": total_pages,
        "halaman_usable": usable_pages,
        "coverage_ratio": round(usable_pages / max(total_pages, 1), 4),
        "pages_with_tables": pages_tables,
        "pages_with_pictures": pages_pictures,
    }
    print("\n" + "=" * 64)
    print("RINGKASAN EKSTRAKSI (dari quality_report_tipe_2)")
    print("=" * 64)
    print(f"  Total halaman        : {total_pages:,}")
    print(f"  Halaman usable       : {usable_pages:,} ({usable_pages/max(total_pages,1)*100:.1f}%)")
    print(f"  Halaman dengan tabel : {pages_tables:,}")
    print(f"  Halaman dengan gambar: {pages_pictures:,}")
 
save_json(final_report, OUTPUT_REPORT_DIR / "laporan_chunking_bab4.json")
print(f"\nLaporan Bab 4 : {OUTPUT_REPORT_DIR / 'laporan_chunking_bab4.json'}")
print(f"Jejak eksklusi: {OUTPUT_REPORT_DIR / 'excluded_blocks.jsonl'}")
print(f"\nSELESAI. {total_chunks:,} chunks siap untuk NB07 (FAISS).")
 


RINGKASAN CHUNKING

Total chunks RAG : 23,414

Distribusi chunk_type:
  text_only   :  22092 (94.4%)
  figure_rich :    643 (2.7%)
  table_rich  :    320 (1.4%)
  mixed_rich  :    153 (0.7%)
  toc_table   :    121 (0.5%)
  toc_figure  :     85 (0.4%)

Provenance fallback (source_pages tidak presisi): 332 (1.4%)

Section disisihkan (total 4025):
  garbage_chunk     : 3333
  reference         :  459
  front_matter      :  225
  ocr_broken        :    8

EVALUASI GARBAGE_CHUNK (3333 chunk)

  Per chunk_type: {'text_only': 3330, 'figure_rich': 3}
  Punya tabel/gambar di section (suspek valid): 12
  Word count: min=1 median=5 max=607

  30 contoh teks garbage:
    [ 4w tbl=0] 'WASIL SPAK D000 oat'
    [ 2w tbl=0] 'HASIL SPAK2020-2024'
    [ 6w tbl=0] 'BAB I SEJARAH KORUPSI DI INDONESIA'
    [ 3w tbl=0] '1.1 Latar Belakang'
    [ 1w tbl=0] 'b.'
    [ 6w tbl=0] '1.4.1 IPAK Menurut Dimensi dan Subdimensi'
    [ 5w tbl=0] 'A. Perkembangan IPAK Menurut Wilayah'
    [ 6w tbl=0] 'B. Perkembangan 